# LineFormer probe — batch chart→data on Colab GPU

**Goal:** judge published-LineFormer quality on OUR catalysis figures before
investing in a standalone rewrite. Zero cost: free T4 GPU.

**You need:** `figures/lineformer_probe.zip` from the repo machine (30 images:
10 figures as full + single-panel crops).

Runtime → Change runtime type → **T4 GPU**, then run cells top to bottom.

In [ ]:
# 1. GPU check
import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| GPU:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU — set Runtime > Change runtime type > T4 GPU'

In [ ]:
# 2. Clone the pipeline repo (LineFormer + ChartDete wrapper)
%cd /content
!rm -rf extract-line-chart-data
!git clone -q https://github.com/tdsone/extract-line-chart-data.git
%cd extract-line-chart-data
!ls

In [ ]:
# 3. Install — mmcv/mmdet pinned to THIS Colab's torch up front.
#    This is the fragile step; expect ~5-10 min. Errors about 'mmcv version'
#    at import time are what cell 4 is for.
import torch
cu = 'cu' + torch.version.cuda.replace('.', '')
tv = torch.__version__.split('+')[0]
print(f'pinning mmcv for torch{tv}/{cu}')
!pip install -q openmim mmengine
!mim install -q "mmcv>=2.0.0,<2.2.0"
!pip install -q -e ".[local]"
!bash setup_local_env.sh || echo '--- setup_local_env.sh reported errors: read them before continuing ---'

### 4. If the install fights the current torch
MMDetection-era code may need an older torch. Nuclear option (slow, ~5 min):
```python
!pip install -q torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu121
!pip install -q mmcv==2.1.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html
```
then **Runtime → Restart session** and re-run from cell 2 (skip the pin in cell 3).

In [ ]:
# 5. Checkpoints present? (setup_local_env.sh usually fetches them;
#    if empty, follow the repo README Google Drive links into the listed paths)
!find . -name '*.pth' -maxdepth 4 2>/dev/null
import glob
assert glob.glob('**/*.pth', recursive=True), 'No checkpoints — fetch per repo README before continuing'

In [ ]:
# 6. Upload the probe zip (figures/lineformer_probe.zip from the repo machine)
import os, zipfile
from google.colab import files
os.makedirs('input', exist_ok=True)
up = files.upload()                       # pick lineformer_probe.zip
zname = next(iter(up))
with zipfile.ZipFile(zname) as z:
    z.extractall('input')
pngs = [f for f in os.listdir('input') if f.endswith('.png')]
print(f'input/: {len(pngs)} images')

In [ ]:
# 7. Batch extract (single-panel crops are the intended input; the *__full*
#    images are there to see how it copes with composites)
from plextract import extract
extract(input_dir='input', output_dir='output', backend='local')
!find output -name '*.json' | head -20

In [ ]:
# 8. Zip everything (traces + any overlay renders) and download
import shutil
shutil.make_archive('lineformer_probe_results', 'zip', 'output')
from google.colab import files
files.download('lineformer_probe_results.zip')
print('drop this zip back on the repo machine: figures/lineformer_probe_results.zip')

## What happens next (on the repo machine)
Claude fuses the LineFormer pixel traces with the existing axis-calibration
layer and rebuilds the verification HTML — LineFormer line extraction
side-by-side with the originals and with the CV reader, so the quality call
is made by eye on our own figures.

**Judging criteria:** (1) right number of curves per panel — especially on the
GRAYSCALE marker figures (catcom_2009, jcat_2015.01) where the CV reader is
blind; (2) traces follow the printed curves through crossings; (3) composite
(*__full*) behaviour vs pre-split panels.